# 01b - Pull public data, memory-safe version (use this for Community Forensics)

**Why this exists:** notebook 01 ran out of memory on Community Forensics (its files are stored in very large blocks).
This notebook runs the group's tested script, which checks free memory before opening a block, saves as it goes,
restarts itself between attempts, and matches real and AI images by file type and size.

**Rules:** CPU runtime. Run cells one at a time, top to bottom. Edit ONLY cell 4 (three lines).
Do PILOT = True first, then PILOT = False for the real pull.

In [ ]:
# CELL 1 - install (about 1 minute). If Colab asks to restart the session, do it, then continue with cell 2.
!pip -q install -U datasets "pyarrow<24" huggingface_hub psutil

In [ ]:
# CELL 2 - connect Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# CELL 3 - download the project code (the pull script lives in src/)
import os, subprocess
if os.path.isdir("/content/repo"):
    subprocess.run(["rm", "-rf", "/content/repo"])
r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/andrinawatsemba/DEEPFAKE_CV_INVESTIGATION.git", "/content/repo"])
assert os.path.exists("/content/repo/src/pull_my_share_NC_modified.py"), "script not found in the repo"
print("Code ready.")

In [ ]:
# CELL 4 - EDIT THESE THREE LINES ONLY
MEMBER = "SM"      # your pull code: GW, AW, NG, NC, SM or PN
PILOT = True       # True = tiny test (10 per class). Change to False for the real pull.
ROOT = "/content/drive/MyDrive/Deepfake Detection Project/datasets"   # the project data folder

In [ ]:
# CELL 5 - run the pull (it prints progress; keep this tab open)
import re, os
assert os.path.isdir(ROOT) and os.path.isdir(os.path.join(ROOT, "metadata")), (
    "ROOT is not the shared data folder (no metadata/ folder inside). Fix the ROOT line in cell 4. "
    "Do NOT continue: the script would create a private folder in your own Drive.")
src = open("/content/repo/src/pull_my_share_NC_modified.py").read()
src = re.sub(r'(?m)^MEMBER = .*$', f'MEMBER = "{MEMBER}"', src, count=1)
src = re.sub(r'(?m)^PILOT = .*$', f'PILOT = {PILOT}', src, count=1)
src = re.sub(r'(?m)^ROOT = .*$', f'ROOT = {ROOT!r}', src, count=1)
open("/content/pull_run.py", "w").write(src)
print("Running as", MEMBER, "| pilot:", PILOT)
!python -u /content/pull_run.py

In [ ]:
# CELL 6 - what did I get?  Both classes equal, many generators, and (for Community Forensics) all PNG.
import glob, pandas as pd
for f in sorted(glob.glob(os.path.join(ROOT, "metadata", f"public_*_{MEMBER}.csv"))):
    d = pd.read_csv(f)
    print("\n", os.path.basename(f), "rows =", len(d), d.label.value_counts().to_dict())
    print("formats:", d.format.value_counts().to_dict(), "| generators:", d.generator.nunique())

## Share your result with the group (only after the REAL pull, PILOT = False)
Images stay on Drive. Upload the small CSV file(s) to GitHub: repo > `metadata/` > **Add file > Upload files**, choose
**Create a new branch ... start a pull request**, name the branch `<code>/data-pull`.

In [ ]:
# CELL 7 - download your CSV file(s) to upload to GitHub
from google.colab import files
for f in sorted(glob.glob(os.path.join(ROOT, "metadata", f"public_*_{MEMBER}.csv"))):
    files.download(f)